In [ ]:
# ============================================================
# 0. Imports
# ============================================================

from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from difflib import SequenceMatcher

import hashlib
import json
import math
import platform
import re
import sys
import unicodedata

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D11"

DOCUMENT_NAME = (
    "Siyaram Silk Mills Limited — Investor Presentation Q4 & FY24"
)

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"

INPUT_REPRESENTATION = (
    "Complete deterministically normalised page-aware structural Markdown"
)

EXPECTED_SOURCE_SHA256 = (
    "604c536562921441733fa3a2b95d3cd43da9aa9f9a665ced87e59c88c5bdd952"
)

EXPECTED_RECORD_COUNT = 199

EXPECTED_CATEGORY_COUNTS = {
    "Presentation metadata": 3,
    "Management commentary": 17,
    "Quarterly business performance": 45,
    "Profit and loss statement": 102,
    "Company profile": 8,
    "Corporate timeline": 17,
    "Operational footprint": 7,
}

FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location",
]

VALUE_ALLOWED_TYPES = (
    str,
    int,
    float,
    type(None),
)

# Frozen from final D11 Branch A validation.
IDENTITY_FIELDS = [
    "Category",
    "Topic",
    "Reporting Period",
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Value",
    "Unit",
    "Source Location",
]

DIAGNOSTIC_FIELDS = [
    "Description",
]

EXPECTED_SOURCE_PAGES = {
    1, 2, 4, 5, 6, 8, 9, 10
}

EXPECTED_QUALIFIED_VALUES = {
    "800+",
    "~100",
    "245+",
    "~1.85",
    "~4.5",
    "5 and counting",
}

OUTPUT_DIR = Path(
    "outputs_D11_validation_C_revised"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected records:", EXPECTED_RECORD_COUNT)
print("Identity fields:", IDENTITY_FIELDS)
print("Primary correctness fields:", PRIMARY_CORRECTNESS_FIELDS)
print("Output directory:", OUTPUT_DIR)

Document: D11
Branch: C
Expected records: 199
Identity fields: ['Category', 'Topic', 'Reporting Period']
Primary correctness fields: ['Value', 'Unit', 'Source Location']
Output directory: outputs_D11_validation_C_revised


In [ ]:
# ============================================================
# 2. Upload canonical validation inputs
# ============================================================
# Required:
#   1) D11_reference_values.csv
#   2) D11_branch_C_structure_check.json
#   3) D11_branch_C_experiment_metadata.json
#   4) D11_branch_C_normalisation_check.json
#   5) D11_branch_C_experiment_summary.json
#
# Required only when the Branch C run is content-evaluable:
#   6) D11_branch_C_parsed_extraction.json

print(
    "Upload:\n"
    "1. D11_reference_values.csv\n"
    "2. D11_branch_C_structure_check.json\n"
    "3. D11_branch_C_experiment_metadata.json\n"
    "4. D11_branch_C_normalisation_check.json\n"
    "5. D11_branch_C_experiment_summary.json\n"
    "6. D11_branch_C_parsed_extraction.json if it was created"
)

uploaded = files.upload()
uploaded_paths = [Path(name) for name in uploaded]

csv_paths = [p for p in uploaded_paths if p.suffix.lower() == ".csv"]
json_paths = [p for p in uploaded_paths if p.suffix.lower() == ".json"]

if len(csv_paths) != 1:
    raise ValueError(
        "Upload exactly one CSV file: D11_reference_values.csv."
    )

REFERENCE_PATH = csv_paths[0]

EXTRACTION_PATH = None
STRUCTURE_PATH = None
METADATA_PATH = None
NORMALISATION_INTEGRITY_PATH = None
EXPERIMENT_SUMMARY_PATH = None


def canonical_filename(path):
    return path.name.casefold().replace(" ", "_")


# First pass: canonical filename patterns.
for path in json_paths:
    filename = canonical_filename(path)

    if "d11_branch_c_parsed_extraction" in filename:
        EXTRACTION_PATH = path
        continue

    if "d11_branch_c_structure_check" in filename:
        STRUCTURE_PATH = path
        continue

    if (
        "d11_branch_c_experiment_metadata" in filename
        and "_pre" not in filename
    ):
        METADATA_PATH = path
        continue

    if (
        "d11_branch_c_normalisation_check" in filename
        or "d11_branch_c_normalization_check" in filename
    ):
        NORMALISATION_INTEGRITY_PATH = path
        continue

    if "d11_branch_c_experiment_summary" in filename:
        EXPERIMENT_SUMMARY_PATH = path
        continue


# Second pass: content-based fallback.
for path in json_paths:
    with path.open("r", encoding="utf-8-sig") as file:
        obj = json.load(file)

    if not isinstance(obj, dict):
        continue

    if (
        EXPERIMENT_SUMMARY_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "parsed_extraction_created" in obj
        and "records_evaluable" in obj
        and "validation_status" in obj
    ):
        EXPERIMENT_SUMMARY_PATH = path
        continue

    if (
        NORMALISATION_INTEGRITY_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and obj.get("parent_branch") == "B"
        and "normalisation_integrity_passed" in obj
        and "parent_equivalence_passed" in obj
    ):
        NORMALISATION_INTEGRITY_PATH = path
        continue

    if (
        METADATA_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "source_sha256" in obj
        and "raw_response_sha256" in obj
        and "structure_check_file" in obj
    ):
        METADATA_PATH = path
        continue

    if (
        STRUCTURE_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "structure_valid" in obj
        and "records_evaluable" in obj
        and "validation_status" not in obj
    ):
        STRUCTURE_PATH = path
        continue

    if (
        EXTRACTION_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and isinstance(obj.get("records"), list)
    ):
        EXTRACTION_PATH = path
        continue


for label, path in {
    "structure check": STRUCTURE_PATH,
    "experiment metadata": METADATA_PATH,
    "normalisation check": NORMALISATION_INTEGRITY_PATH,
    "experiment summary": EXPERIMENT_SUMMARY_PATH,
}.items():
    if path is None:
        raise ValueError(f"Could not identify required {label} file.")


experiment_summary = json.loads(
    EXPERIMENT_SUMMARY_PATH.read_text(encoding="utf-8")
)

content_evaluable = bool(
    experiment_summary.get("records_evaluable")
    and experiment_summary.get("parsed_extraction_created")
)

if content_evaluable and EXTRACTION_PATH is None:
    raise ValueError(
        "This Branch C run is content-evaluable, so "
        "D11_branch_C_parsed_extraction.json is required."
    )

if not content_evaluable:
    raise ValueError(
        "This D11 Branch C run is not content-evaluable. "
        "Do not calculate record- or field-level metrics. "
        "Send the experiment summary and use a non-evaluable "
        "validation pathway, as in D9."
    )


print("\nIdentified D11 Branch C validation inputs:")
print("Reference:", REFERENCE_PATH.name)
print("Parsed extraction:", EXTRACTION_PATH.name)
print("Structure check:", STRUCTURE_PATH.name)
print("Experiment metadata:", METADATA_PATH.name)
print("Normalisation check:", NORMALISATION_INTEGRITY_PATH.name)
print("Experiment summary:", EXPERIMENT_SUMMARY_PATH.name)
print("Content evaluable:", content_evaluable)

Upload:
1. D11_reference_values.csv
2. D11_branch_C_structure_check.json
3. D11_branch_C_experiment_metadata.json
4. D11_branch_C_normalisation_check.json
5. D11_branch_C_experiment_summary.json
6. D11_branch_C_parsed_extraction.json if it was created


Saving D11_branch_C_structure_check.json to D11_branch_C_structure_check.json
Saving D11_branch_C_parsed_extraction.json to D11_branch_C_parsed_extraction.json
Saving D11_branch_C_normalisation_check.json to D11_branch_C_normalisation_check.json
Saving D11_branch_C_experiment_summary.json to D11_branch_C_experiment_summary.json
Saving D11_branch_C_experiment_metadata.json to D11_branch_C_experiment_metadata.json
Saving D11_reference_values.csv to D11_reference_values.csv

Identified D11 Branch C validation inputs:
Reference: D11_reference_values.csv
Parsed extraction: D11_branch_C_parsed_extraction.json
Structure check: D11_branch_C_structure_check.json
Experiment metadata: D11_branch_C_experiment_metadata.json
Normalisation check: D11_branch_C_normalisation_check.json
Experiment summary: D11_branch_C_experiment_summary.json
Content evaluable: True


In [ ]:
# ============================================================
# 3. Hashing and load inputs
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


REFERENCE_SHA256 = sha256_file(REFERENCE_PATH)
EXTRACTION_SHA256 = sha256_file(EXTRACTION_PATH)
STRUCTURE_SHA256 = sha256_file(STRUCTURE_PATH)
METADATA_SHA256 = sha256_file(METADATA_PATH)
NORMALISATION_INTEGRITY_SHA256 = sha256_file(
    NORMALISATION_INTEGRITY_PATH
)
EXPERIMENT_SUMMARY_SHA256 = sha256_file(
    EXPERIMENT_SUMMARY_PATH
)


# ------------------------------------------------------------
# Fixed Stage 1 reference
# ------------------------------------------------------------

reference_df = pd.read_csv(
    REFERENCE_PATH,
    dtype=object,
    keep_default_na=True,
)

reference_df = reference_df.where(
    pd.notna(reference_df),
    None,
)


def restore_reference_value(value):
    if value is None or pd.isna(value):
        return None

    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return value

    text = str(value).strip()

    if (
        text.startswith("~")
        or text.endswith("+")
        or "and counting" in text.casefold()
    ):
        return text

    if re.fullmatch(
        r"-?\d+(?:\.\d+)?",
        text,
    ):
        number = float(text)

        return (
            int(number)
            if number.is_integer()
            else number
        )

    return text


reference_df["Value"] = (
    reference_df["Value"]
    .map(restore_reference_value)
)


# ------------------------------------------------------------
# Canonical Branch C extraction
# ------------------------------------------------------------

parsed_extraction = json.loads(
    EXTRACTION_PATH.read_text(
        encoding="utf-8"
    )
)

top_level_object_valid = isinstance(
    parsed_extraction,
    dict,
)

document_id_correct = (
    top_level_object_valid
    and parsed_extraction.get("document_id")
    == DOCUMENT_ID
)

branch_correct = (
    top_level_object_valid
    and parsed_extraction.get("branch")
    == BRANCH
)

records_is_list = (
    top_level_object_valid
    and isinstance(
        parsed_extraction.get("records"),
        list,
    )
)

if not records_is_list:
    raise ValueError(
        "Canonical Branch C parsed extraction must contain a records list."
    )

extracted_records = parsed_extraction["records"]

extracted_df = pd.DataFrame(
    extracted_records
)


# ------------------------------------------------------------
# Branch C technical/provenance artefacts
# ------------------------------------------------------------

structure_check = json.loads(
    STRUCTURE_PATH.read_text(
        encoding="utf-8"
    )
)

experiment_metadata = json.loads(
    METADATA_PATH.read_text(
        encoding="utf-8"
    )
)

normalisation_integrity = json.loads(
    NORMALISATION_INTEGRITY_PATH.read_text(
        encoding="utf-8"
    )
)


for artefact_name, artefact in {
    "structure check": structure_check,
    "experiment metadata": experiment_metadata,
    "normalisation integrity": normalisation_integrity,
}.items():

    if artefact.get("document_id") != DOCUMENT_ID:
        raise ValueError(
            f"Unexpected {artefact_name} document_id: "
            f"{artefact.get('document_id')}"
        )

    if artefact.get("branch") != BRANCH:
        raise ValueError(
            f"Unexpected {artefact_name} branch: "
            f"{artefact.get('branch')}"
        )


branch_c_structure_valid = bool(
    structure_check.get("structure_valid")
)

parsed_extraction_hash_matches_metadata = (
    experiment_metadata.get("parsed_extraction_sha256")
    == EXTRACTION_SHA256
)

source_hash_matches_stage_1 = (
    experiment_metadata.get("source_sha256")
    == EXPECTED_SOURCE_SHA256
)

parent_equivalence_passed = bool(
    normalisation_integrity.get(
        "parent_equivalence_passed",
        False,
    )
)

normalisation_integrity_passed = bool(
    normalisation_integrity.get(
        "normalisation_integrity_passed",
        False,
    )
)


print("Reference SHA-256:", REFERENCE_SHA256)
print("Extraction SHA-256:", EXTRACTION_SHA256)
print("Branch C structure valid:", branch_c_structure_valid)

print(
    "Parsed extraction hash matches metadata:",
    parsed_extraction_hash_matches_metadata,
)

print(
    "Source hash matches Stage 1:",
    source_hash_matches_stage_1,
)

print(
    "Parent B equivalence passed:",
    parent_equivalence_passed,
)

print(
    "Normalisation integrity passed:",
    normalisation_integrity_passed,
)


if not parsed_extraction_hash_matches_metadata:
    raise AssertionError(
        "Parsed extraction does not match Branch C experiment metadata."
    )

if not source_hash_matches_stage_1:
    raise AssertionError(
        "Branch C did not use the fixed Stage 1 D11 source."
    )

if not parent_equivalence_passed:
    raise AssertionError(
        "Branch C parent-B equivalence did not pass."
    )

if not normalisation_integrity_passed:
    raise AssertionError(
        "Branch C normalisation integrity did not pass."
    )

Reference SHA-256: e353b131db3828565f05653b040af46a0c864835e710fec8953e79c0b22eda41
Extraction SHA-256: be21038b4650b638179866ae33cb91527d82013615ce49719260776c3f689dc4
Branch C structure valid: True
Parsed extraction hash matches metadata: True
Source hash matches Stage 1: True
Parent B equivalence passed: True
Normalisation integrity passed: True


In [ ]:
# ============================================================
# 5. Schema, types and content diagnostics
# ============================================================
# IMPORTANT:
# - Structural/type validity is kept separate from extraction completeness.
# - Reporting Period may be null in the preserved model output and still
#   remain structurally type-valid. If the fixed reference expects a period,
#   the null value is penalised later through identity alignment/correctness.
# - The Stage 1 reference itself remains subject to the stricter frozen
#   reference requirements.

reference_schema_exact = (
    reference_df.columns.tolist()
    == FIELDS
)

extraction_schema_exact = (
    extracted_df.columns.tolist()
    == FIELDS
)


REFERENCE_REQUIRED_STRING_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location",
]

EXTRACTION_REQUIRED_STRING_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Source Location",
]

EXTRACTION_NULLABLE_STRING_FIELDS = [
    "Reporting Period",
]


def validate_reference_types(records):
    issues = []

    for index, record in enumerate(records):

        if not isinstance(record, dict):
            issues.append({
                "dataset": "Reference",
                "record_index": index,
                "field": None,
                "issue": "Record is not an object",
            })
            continue

        if set(record.keys()) != set(FIELDS):
            issues.append({
                "dataset": "Reference",
                "record_index": index,
                "field": None,
                "issue": "Field set differs from schema",
                "observed_fields": list(record.keys()),
            })

        for field in REFERENCE_REQUIRED_STRING_FIELDS:
            value = record.get(field)

            if (
                value is None
                or not isinstance(value, str)
                or not value.strip()
            ):
                issues.append({
                    "dataset": "Reference",
                    "record_index": index,
                    "field": field,
                    "issue": "Expected non-empty string",
                    "observed_type":
                        None if value is None else type(value).__name__,
                })

        value = record.get("Value")

        if (
            isinstance(value, bool)
            or not isinstance(
                value,
                VALUE_ALLOWED_TYPES,
            )
        ):
            issues.append({
                "dataset": "Reference",
                "record_index": index,
                "field": "Value",
                "issue": "Expected string, number or null",
                "observed_type": type(value).__name__,
            })

    return issues


def validate_extraction_types(records):
    issues = []

    for index, record in enumerate(records):

        if not isinstance(record, dict):
            issues.append({
                "dataset": "Extraction",
                "record_index": index,
                "field": None,
                "issue": "Record is not an object",
            })
            continue

        if set(record.keys()) != set(FIELDS):
            issues.append({
                "dataset": "Extraction",
                "record_index": index,
                "field": None,
                "issue": "Field set differs from schema",
                "observed_fields": list(record.keys()),
            })

        # Fields required to contain non-empty strings.
        for field in EXTRACTION_REQUIRED_STRING_FIELDS:
            value = record.get(field)

            if (
                value is None
                or not isinstance(value, str)
                or not value.strip()
            ):
                issues.append({
                    "dataset": "Extraction",
                    "record_index": index,
                    "field": field,
                    "issue": "Expected non-empty string",
                    "observed_type":
                        None if value is None else type(value).__name__,
                })

        # Reporting Period is structurally allowed to be string or null.
        for field in EXTRACTION_NULLABLE_STRING_FIELDS:
            value = record.get(field)

            if (
                value is not None
                and not isinstance(value, str)
            ):
                issues.append({
                    "dataset": "Extraction",
                    "record_index": index,
                    "field": field,
                    "issue": "Expected string or null",
                    "observed_type": type(value).__name__,
                })

        value = record.get("Value")

        if (
            isinstance(value, bool)
            or not isinstance(
                value,
                VALUE_ALLOWED_TYPES,
            )
        ):
            issues.append({
                "dataset": "Extraction",
                "record_index": index,
                "field": "Value",
                "issue": "Expected string, number or null",
                "observed_type": type(value).__name__,
            })

    return issues


reference_type_issues = validate_reference_types(
    reference_df.to_dict("records")
)

extraction_type_issues = validate_extraction_types(
    extracted_records
)

reference_types_valid = (
    len(reference_type_issues) == 0
)

extraction_types_valid = (
    len(extraction_type_issues) == 0
)


reference_record_count_valid = (
    len(reference_df)
    == EXPECTED_RECORD_COUNT
)

extraction_record_count_valid = (
    len(extracted_df)
    == EXPECTED_RECORD_COUNT
)


reference_category_counts = (
    reference_df["Category"]
    .value_counts()
    .to_dict()
)

extraction_category_counts = (
    extracted_df["Category"]
    .value_counts()
    .to_dict()
)

reference_category_counts_valid = (
    reference_category_counts
    == EXPECTED_CATEGORY_COUNTS
)

extraction_category_counts_valid = (
    extraction_category_counts
    == EXPECTED_CATEGORY_COUNTS
)


# Schema validity does NOT depend on record-count or category-count
# completeness. Those are content/scope outcomes.
schema_validity = all([
    top_level_object_valid,
    document_id_correct,
    branch_correct,
    records_is_list,
    extraction_schema_exact,
    extraction_types_valid,
    branch_c_structure_valid,
])


print("Reference schema exact:", reference_schema_exact)
print("Extraction schema exact:", extraction_schema_exact)
print("Reference types valid:", reference_types_valid)
print("Extraction types valid:", extraction_types_valid)
print("Schema validity:", schema_validity)
print("Reference count:", len(reference_df))
print("Extraction count:", len(extracted_df))
print("Reference category counts:", reference_category_counts)
print("Extraction category counts:", extraction_category_counts)

if reference_type_issues:
    print("\nReference type issues:")
    display(pd.DataFrame(reference_type_issues))

if extraction_type_issues:
    print("\nExtraction type issues:")
    display(pd.DataFrame(extraction_type_issues))

Reference schema exact: True
Extraction schema exact: True
Reference types valid: True
Extraction types valid: True
Schema validity: True
Reference count: 199
Extraction count: 163
Reference category counts: {'Profit and loss statement': 102, 'Quarterly business performance': 45, 'Management commentary': 17, 'Corporate timeline': 17, 'Company profile': 8, 'Operational footprint': 7, 'Presentation metadata': 3}
Extraction category counts: {'Profit and loss statement': 102, 'Corporate timeline': 17, 'Management commentary': 17, 'Quarterly business performance': 9, 'Company profile': 8, 'Operational footprint': 7, 'Presentation metadata': 3}


In [ ]:
# ============================================================
# 6. Confirm current Stage 1 D11 reference semantics
# ============================================================

def extract_page(value):
    if value is None:
        return None

    match = re.fullmatch(
        r"PDF page\s+(\d+)",
        str(value).strip(),
        flags=re.IGNORECASE,
    )

    if not match:
        return None

    return int(
        match.group(1)
    )


observed_source_pages = {
    page
    for page in (
        reference_df["Source Location"]
        .map(extract_page)
    )
    if page is not None
}

observed_qualified_values = {
    str(value)
    for value in reference_df["Value"]
    if str(value) in EXPECTED_QUALIFIED_VALUES
}

identity_duplicate_count = int(
    reference_df.duplicated(
        subset=IDENTITY_FIELDS,
        keep=False,
    ).sum()
)

timeline_year_units_valid = all([
    reference_df.loc[
        (reference_df["Category"] == "Corporate timeline")
        & (reference_df["Topic"] == "Established"),
        "Unit",
    ].eq("year").all(),

    reference_df.loc[
        (reference_df["Category"] == "Corporate timeline")
        & (reference_df["Topic"] == "Public listing"),
        "Unit",
    ].eq("year").all(),
])

reference_semantic_checks = {
    "reference_schema_exact":
        bool(reference_schema_exact),

    "reference_types_valid":
        bool(reference_types_valid),

    "reference_record_count_valid":
        bool(reference_record_count_valid),

    "reference_category_counts_valid":
        bool(reference_category_counts_valid),

    "reference_identity_unique":
        identity_duplicate_count == 0,

    "source_page_coverage_valid":
        observed_source_pages
        == EXPECTED_SOURCE_PAGES,

    "qualified_values_preserved":
        observed_qualified_values
        == EXPECTED_QUALIFIED_VALUES,

    "timeline_year_units_valid":
        bool(
            timeline_year_units_valid
        ),

    "no_reference_records_from_divider_pages":
        not bool(
            observed_source_pages
            & {3, 7}
        ),
}

reference_semantics_valid = all(
    reference_semantic_checks.values()
)

print(
    json.dumps(
        reference_semantic_checks,
        ensure_ascii=False,
        indent=2,
    )
)

print(
    "Current D11 reference semantics valid:",
    reference_semantics_valid,
)

if not reference_semantics_valid:
    raise AssertionError(
        "The supplied D11 reference does not match the "
        "current frozen Stage 1 D11 reference semantics."
    )

{
  "reference_schema_exact": true,
  "reference_types_valid": true,
  "reference_record_count_valid": true,
  "reference_category_counts_valid": true,
  "reference_identity_unique": true,
  "source_page_coverage_valid": true,
  "qualified_values_preserved": true,
  "timeline_year_units_valid": true,
  "no_reference_records_from_divider_pages": true
}
Current D11 reference semantics valid: True


In [ ]:
# ============================================================
# 7. Comparison-only normalisation
# ============================================================

def normalise_text(value):
    if value is None:
        return ""

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    text = (
        text
        .replace("\u00a0", " ")
        .replace("\u2007", " ")
        .replace("\u202f", " ")
        .replace("—", "-")
        .replace("–", "-")
        .replace("‑", "-")
        .replace("’", "'")
        .replace("“", '"')
        .replace("”", '"')
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text.casefold()


def normalise_topic(value):
    return normalise_text(
        value
    )


PERIOD_EQUIVALENCE = {
    "q4fy24": "q4 fy24",
    "q4fy23": "q4 fy23",
    "q3fy24": "q3 fy24",
    "q4 & fy24": "q4 & fy24",
    "q4 and fy24": "q4 & fy24",
    "31 march 2024": "2024-03-31",
    "march 31, 2024": "2024-03-31",
    "march 31 2024": "2024-03-31",
    "2024-03-31": "2024-03-31",
}


def canonical_period(value):
    text = normalise_text(
        value
    )

    return PERIOD_EQUIVALENCE.get(
        text,
        text,
    )


UNIT_EQUIVALENCE = {
    "text": "text",
    "rs. mn": "rs mn",
    "rs mn": "rs mn",
    "₹ mn": "rs mn",
    "₹ in mn": "rs mn",
    "rs. crores": "rs crores",
    "rs crores": "rs crores",
    "rs. per share": "rs per share",
    "rs per share": "rs per share",
    "rs.": "rs",
    "rs": "rs",
    "%": "percent",
    "percentage": "percent",
    "percent": "percent",
    "store": "stores",
    "stores": "stores",
    "distributor": "distributors",
    "distributors": "distributors",
    "mn meter": "mn meters",
    "mn meters": "mn meters",
    "mn piece": "mn pieces",
    "mn pieces": "mn pieces",
    "mn customer": "mn customers",
    "mn customers": "mn customers",
    "l sq ft": "l sqft",
    "l sqft": "l sqft",
    "year": "year",
}


def canonical_unit(value):
    text = normalise_text(
        value
    )

    return UNIT_EQUIVALENCE.get(
        text,
        text,
    )


def parse_numeric(value):
    if isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    if not isinstance(value, str):
        return None

    text = value.strip().replace(
        ",",
        "",
    )

    if re.fullmatch(
        r"-?\d+(?:\.\d+)?",
        text,
    ):
        try:
            return float(text)
        except ValueError:
            return None

    return None


def has_source_qualifier(value):
    if not isinstance(value, str):
        return False

    text = value.strip().casefold()

    return (
        text.startswith("~")
        or text.endswith("+")
        or "and counting" in text
    )


def value_equal(reference_value, extracted_value):
    if reference_value is None and extracted_value is None:
        return True

    # Qualified values must remain textual and preserve the qualifier.
    if (
        has_source_qualifier(reference_value)
        or has_source_qualifier(extracted_value)
    ):
        return (
            normalise_text(reference_value)
            == normalise_text(extracted_value)
        )

    ref_numeric = parse_numeric(
        reference_value
    )

    ext_numeric = parse_numeric(
        extracted_value
    )

    if (
        ref_numeric is not None
        and ext_numeric is not None
    ):
        return math.isclose(
            ref_numeric,
            ext_numeric,
            rel_tol=0.0,
            abs_tol=1e-12,
        )

    # Conservative first-run rule for textual values:
    # exact after deterministic Unicode/whitespace normalisation.
    return (
        normalise_text(reference_value)
        == normalise_text(extracted_value)
    )


def unit_equal(reference_value, extracted_value):
    return (
        canonical_unit(reference_value)
        == canonical_unit(extracted_value)
    )


def source_location_equal(reference_value, extracted_value):
    return (
        extract_page(reference_value)
        == extract_page(extracted_value)
        and extract_page(reference_value)
        is not None
    )


def description_exact(reference_value, extracted_value):
    return (
        normalise_text(reference_value)
        == normalise_text(extracted_value)
    )

In [ ]:
# ============================================================
# 8. Controlled one-to-one identity alignment
# ============================================================
# IMPORTANT:
# This is the exact D11 alignment specification established
# and frozen in the revised Branch A validation.
#
# HARD BLOCK:
#   Category
#
# IDENTITY EVIDENCE:
#   Topic
#   Reporting Period
#
# EXCLUDED FROM ALIGNMENT:
#   Value
#   Unit
#   Description
#   Source Location
#
# Step 1:
#   Assign strict Category + Topic + Reporting Period identities.
#
# Step 2:
#   For remaining records only, perform one-to-one matching
#   within Category using Topic + Reporting Period similarity.
#
# No Branch-B-specific matching rules are introduced.


def lexical_similarity(left, right):
    left_norm = normalise_text(left)
    right_norm = normalise_text(right)

    if left_norm == right_norm:
        return 1.0

    if not left_norm or not right_norm:
        return 0.0

    return SequenceMatcher(
        None,
        left_norm,
        right_norm,
    ).ratio()


def topic_similarity(left, right):
    left_norm = normalise_topic(left)
    right_norm = normalise_topic(right)

    if left_norm == right_norm:
        return 1.0

    if not left_norm or not right_norm:
        return 0.0

    return SequenceMatcher(
        None,
        left_norm,
        right_norm,
    ).ratio()


def period_similarity(left, right):
    left_period = canonical_period(left)
    right_period = canonical_period(right)

    if left_period == right_period:
        return 1.0

    if left_period is None or right_period is None:
        return 0.0

    return lexical_similarity(
        left_period,
        right_period,
    )


# ------------------------------------------------------------
# Frozen D11 alignment parameters from revised Branch A
# ------------------------------------------------------------

ALIGNMENT_WEIGHTS = {
    "topic": 0.75,
    "reporting_period": 0.25,
}

ALIGNMENT_SCORE_THRESHOLD = 0.60


def alignment_score(
    reference_record,
    extracted_record,
):
    # Category is a hard block.
    if (
        normalise_text(
            reference_record["Category"]
        )
        !=
        normalise_text(
            extracted_record["Category"]
        )
    ):
        return 0.0

    topic_score = topic_similarity(
        reference_record["Topic"],
        extracted_record["Topic"],
    )

    period_score = period_similarity(
        reference_record["Reporting Period"],
        extracted_record["Reporting Period"],
    )

    return (
        ALIGNMENT_WEIGHTS["topic"]
        * topic_score
        +
        ALIGNMENT_WEIGHTS["reporting_period"]
        * period_score
    )


reference_records = (
    reference_df
    .to_dict("records")
)

extraction_records = (
    extracted_df
    .to_dict("records")
)


matches = []

matched_reference_indices = set()
matched_extraction_indices = set()

alignment_diagnostics = []


# ------------------------------------------------------------
# Step 1 — strict one-to-one identities
# ------------------------------------------------------------

def strict_identity_key(record):
    return (
        normalise_text(
            record["Category"]
        ),
        normalise_topic(
            record["Topic"]
        ),
        canonical_period(
            record["Reporting Period"]
        ),
    )


reference_identity_map = defaultdict(list)
extraction_identity_map = defaultdict(list)


for index, record in enumerate(
    reference_records
):
    reference_identity_map[
        strict_identity_key(record)
    ].append(index)


for index, record in enumerate(
    extraction_records
):
    extraction_identity_map[
        strict_identity_key(record)
    ].append(index)


reference_duplicate_identity_count = sum(
    len(indices) - 1
    for indices
    in reference_identity_map.values()
    if len(indices) > 1
)

extraction_duplicate_identity_count = sum(
    len(indices) - 1
    for indices
    in extraction_identity_map.values()
    if len(indices) > 1
)


for key in sorted(
    set(reference_identity_map)
    & set(extraction_identity_map)
):

    ref_indices = reference_identity_map[key]
    ext_indices = extraction_identity_map[key]

    # Assign strict identity directly only when one-to-one.
    if (
        len(ref_indices) == 1
        and len(ext_indices) == 1
    ):
        ref_index = ref_indices[0]
        ext_index = ext_indices[0]

        matches.append({
            "Reference Index":
                ref_index,

            "Extraction Index":
                ext_index,

            "Alignment Rule":
                "Strict Category + Topic + Reporting Period",

            "Alignment Score":
                1.0,

            "Topic Similarity":
                1.0,

            "Reporting Period Similarity":
                1.0,
        })

        matched_reference_indices.add(
            ref_index
        )

        matched_extraction_indices.add(
            ext_index
        )


# ------------------------------------------------------------
# Step 2 — controlled one-to-one fallback within Category
# ------------------------------------------------------------

categories = sorted(
    {
        normalise_text(
            record["Category"]
        )
        for record
        in reference_records
    }
    |
    {
        normalise_text(
            record["Category"]
        )
        for record
        in extraction_records
    }
)


for category in categories:

    remaining_reference_indices = [
        index
        for index, record
        in enumerate(reference_records)
        if (
            index
            not in matched_reference_indices
            and normalise_text(
                record["Category"]
            ) == category
        )
    ]

    remaining_extraction_indices = [
        index
        for index, record
        in enumerate(extraction_records)
        if (
            index
            not in matched_extraction_indices
            and normalise_text(
                record["Category"]
            ) == category
        )
    ]

    if (
        not remaining_reference_indices
        or not remaining_extraction_indices
    ):
        continue


    score_matrix = np.zeros(
        (
            len(
                remaining_reference_indices
            ),
            len(
                remaining_extraction_indices
            ),
        ),
        dtype=float,
    )


    for row_index, ref_index in enumerate(
        remaining_reference_indices
    ):

        reference_record = (
            reference_records[
                ref_index
            ]
        )

        for column_index, ext_index in enumerate(
            remaining_extraction_indices
        ):

            extracted_record = (
                extraction_records[
                    ext_index
                ]
            )

            score_matrix[
                row_index,
                column_index
            ] = alignment_score(
                reference_record,
                extracted_record,
            )


    # Hungarian assignment maximises total identity score.
    row_indices, column_indices = (
        linear_sum_assignment(
            -score_matrix
        )
    )


    for row_index, column_index in zip(
        row_indices,
        column_indices,
    ):

        score = float(
            score_matrix[
                row_index,
                column_index
            ]
        )

        if (
            score
            < ALIGNMENT_SCORE_THRESHOLD
        ):
            continue


        ref_index = (
            remaining_reference_indices[
                row_index
            ]
        )

        ext_index = (
            remaining_extraction_indices[
                column_index
            ]
        )

        reference_record = (
            reference_records[
                ref_index
            ]
        )

        extracted_record = (
            extraction_records[
                ext_index
            ]
        )


        topic_score = topic_similarity(
            reference_record["Topic"],
            extracted_record["Topic"],
        )

        period_score = period_similarity(
            reference_record[
                "Reporting Period"
            ],
            extracted_record[
                "Reporting Period"
            ],
        )


        match = {
            "Reference Index":
                ref_index,

            "Extraction Index":
                ext_index,

            "Alignment Rule":
                "Controlled Category-blocked fallback",

            "Alignment Score":
                score,

            "Topic Similarity":
                topic_score,

            "Reporting Period Similarity":
                period_score,
        }


        matches.append(
            match
        )

        alignment_diagnostics.append(
            match.copy()
        )

        matched_reference_indices.add(
            ref_index
        )

        matched_extraction_indices.add(
            ext_index
        )


# ------------------------------------------------------------
# Final unmatched observations
# ------------------------------------------------------------

missing_reference_indices = sorted(
    set(
        range(
            len(reference_records)
        )
    )
    - matched_reference_indices
)

unsupported_extraction_indices = sorted(
    set(
        range(
            len(extraction_records)
        )
    )
    - matched_extraction_indices
)


strict_alignment_count = sum(
    match[
        "Alignment Rule"
    ].startswith("Strict")
    for match in matches
)

fallback_alignment_count = sum(
    match[
        "Alignment Rule"
    ].startswith("Controlled")
    for match in matches
)


print(
    "Aligned records:",
    len(matches),
)

print(
    "Strict exact alignments:",
    strict_alignment_count,
)

print(
    "Controlled fallback alignments:",
    fallback_alignment_count,
)

print(
    "Missing reference records:",
    len(
        missing_reference_indices
    ),
)

print(
    "Unsupported/unmatched extraction records:",
    len(
        unsupported_extraction_indices
    ),
)

print(
    "Reference duplicate identity count:",
    reference_duplicate_identity_count,
)

print(
    "Extraction duplicate identity count:",
    extraction_duplicate_identity_count,
)

print(
    "Alignment threshold:",
    ALIGNMENT_SCORE_THRESHOLD,
)

print(
    "Alignment weights:",
    ALIGNMENT_WEIGHTS,
)

Aligned records: 147
Strict exact alignments: 86
Controlled fallback alignments: 61
Missing reference records: 52
Unsupported/unmatched extraction records: 16
Reference duplicate identity count: 0
Extraction duplicate identity count: 0
Alignment threshold: 0.6
Alignment weights: {'topic': 0.75, 'reporting_period': 0.25}


In [ ]:
# ============================================================
# 9. Field comparison
# ============================================================

comparison_rows = []

for match in matches:

    reference_record = reference_records[
        match["Reference Index"]
    ]

    extracted_record = extraction_records[
        match["Extraction Index"]
    ]

    row = {
        "Reference Index":
            match["Reference Index"],

        "Extraction Index":
            match["Extraction Index"],

        "Alignment Rule":
            match["Alignment Rule"],

        "Alignment Score":
            match.get(
                "Alignment Score",
                1.0,
            ),

        "Topic Similarity":
            match.get(
                "Topic Similarity",
                1.0,
            ),

        "Reporting Period Similarity":
            match.get(
                "Reporting Period Similarity",
                1.0,
            ),
    }

    correctness = {
        "Category":
            normalise_text(
                reference_record["Category"]
            )
            == normalise_text(
                extracted_record["Category"]
            ),

        "Topic":
            normalise_topic(
                reference_record["Topic"]
            )
            == normalise_topic(
                extracted_record["Topic"]
            ),

        "Description":
            description_exact(
                reference_record["Description"],
                extracted_record["Description"],
            ),

        "Value":
            value_equal(
                reference_record["Value"],
                extracted_record["Value"],
            ),

        "Unit":
            unit_equal(
                reference_record["Unit"],
                extracted_record["Unit"],
            ),

        "Reporting Period":
            canonical_period(
                reference_record["Reporting Period"]
            )
            == canonical_period(
                extracted_record["Reporting Period"]
            ),

        "Source Location":
            source_location_equal(
                reference_record["Source Location"],
                extracted_record["Source Location"],
            ),
    }

    for field in FIELDS:

        row[
            f"Reference {field}"
        ] = reference_record[field]

        row[
            f"Extracted {field}"
        ] = extracted_record[field]

        row[
            f"{field} Correct"
        ] = bool(
            correctness[field]
        )

    row[
        "Fully Correct Primary Record"
    ] = all(
        correctness[field]
        for field
        in PRIMARY_CORRECTNESS_FIELDS
    )

    comparison_rows.append(
        row
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


missing_records_df = (
    reference_df
    .iloc[
        missing_reference_indices
    ]
    .copy()
)

unsupported_records_df = (
    extracted_df
    .iloc[
        unsupported_extraction_indices
    ]
    .copy()
)

discrepant_records_df = (
    comparison_df.loc[
        ~comparison_df[
            "Fully Correct Primary Record"
        ]
    ]
    .copy()
)


print(
    "Fully correct primary records:",
    int(
        comparison_df[
            "Fully Correct Primary Record"
        ].sum()
    ),
)

print(
    "Primary discrepant records:",
    len(
        discrepant_records_df
    ),
)

Fully correct primary records: 122
Primary discrepant records: 25


In [ ]:
# ============================================================
# 10. Metrics
# ============================================================

aligned_records = len(
    comparison_df
)

fully_correct_records = int(
    comparison_df[
        "Fully Correct Primary Record"
    ].sum()
)

discrepant_records = (
    aligned_records
    - fully_correct_records
)

missing_records = len(
    missing_reference_indices
)

unsupported_records = len(
    unsupported_extraction_indices
)

completeness = (
    aligned_records
    / len(reference_df)
    if len(reference_df)
    else 0.0
)

record_precision_exact = (
    fully_correct_records
    / len(extracted_df)
    if len(extracted_df)
    else 0.0
)

record_recall_exact = (
    fully_correct_records
    / len(reference_df)
    if len(reference_df)
    else 0.0
)

record_f1_exact = (
    2
    * record_precision_exact
    * record_recall_exact
    / (
        record_precision_exact
        + record_recall_exact
    )
    if (
        record_precision_exact
        + record_recall_exact
    )
    else 0.0
)


primary_field_accuracy = {}

for field in PRIMARY_CORRECTNESS_FIELDS:

    primary_field_accuracy[field] = (
        float(
            comparison_df[
                f"{field} Correct"
            ].mean()
        )
        if aligned_records
        else 0.0
    )


diagnostic_field_accuracy = {
    "Description exact":
        (
            float(
                comparison_df[
                    "Description Correct"
                ].mean()
            )
            if aligned_records
            else 0.0
        ),
}


overall_primary_field_accuracy = (
    sum(
        primary_field_accuracy.values()
    )
    / len(
        primary_field_accuracy
    )
)


category_metrics = {}

for category, expected_count in (
    EXPECTED_CATEGORY_COUNTS.items()
):

    ref_subset = reference_df[
        reference_df["Category"]
        == category
    ]

    ext_subset = extracted_df[
        extracted_df["Category"]
        == category
    ]

    aligned_subset = comparison_df[
        comparison_df[
            "Reference Category"
        ]
        == category
    ]

    correct_count = int(
        aligned_subset[
            "Fully Correct Primary Record"
        ].sum()
    )

    precision = (
        correct_count
        / len(ext_subset)
        if len(ext_subset)
        else 0.0
    )

    recall = (
        correct_count
        / len(ref_subset)
        if len(ref_subset)
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    category_metrics[category] = {
        "expected_records":
            int(expected_count),

        "extracted_records":
            int(len(ext_subset)),

        "aligned_records":
            int(len(aligned_subset)),

        "fully_correct_records":
            correct_count,

        "discrepant_records":
            int(
                len(aligned_subset)
                - correct_count
            ),

        "completeness":
            float(
                len(aligned_subset)
                / expected_count
                if expected_count
                else 0.0
            ),

        "record_precision_exact":
            float(precision),

        "record_recall_exact":
            float(recall),

        "record_f1_exact":
            float(f1),
    }


print("Reference records:", len(reference_df))
print("Extracted records:", len(extracted_df))
print("Aligned records:", aligned_records)
print("Fully correct records:", fully_correct_records)
print("Discrepant records:", discrepant_records)
print("Missing records:", missing_records)
print("Unsupported/unmatched records:", unsupported_records)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1_exact, 4))
print(
    "Overall primary field accuracy:",
    round(
        overall_primary_field_accuracy,
        4,
    ),
)
print("Schema valid:", schema_validity)

Reference records: 199
Extracted records: 163
Aligned records: 147
Fully correct records: 122
Discrepant records: 25
Missing records: 52
Unsupported/unmatched records: 16
Completeness: 0.7387
Exact F1: 0.674
Overall primary field accuracy: 0.941
Schema valid: True


In [ ]:
# ============================================================
# 10. Preserve Branch C representation-integrity diagnostics
# ============================================================
# Stage 2 B→C normalisation integrity is reported independently
# from Stage 4 extraction correctness.

representation_integrity = {
    "parent_branch":
        normalisation_integrity.get("parent_branch"),

    "parent_equivalence_passed":
        bool(
            normalisation_integrity.get(
                "parent_equivalence_passed",
                False,
            )
        ),

    "normalisation_integrity_passed":
        bool(
            normalisation_integrity.get(
                "normalisation_integrity_passed",
                False,
            )
        ),

    "page_sequence_preserved":
        normalisation_integrity.get(
            "page_sequence_preserved"
        ),

    "deterministic_representation_verified":
        normalisation_integrity.get(
            "deterministic_representation_verified"
        ),

    "critical_marker_checks":
        normalisation_integrity.get(
            "critical_marker_checks"
        ),

    "all_critical_markers_preserved":
        normalisation_integrity.get(
            "all_critical_markers_preserved"
        ),

    "qualified_source_value_checks":
        normalisation_integrity.get(
            "qualified_source_value_checks"
        ),

    "qualified_source_values_preserved":
        normalisation_integrity.get(
            "qualified_source_values_preserved"
        ),

    "token_preservation":
        normalisation_integrity.get(
            "token_preservation"
        ),

    "tokens_preserved":
        normalisation_integrity.get(
            "tokens_preserved"
        ),

    "divider_pages_3_7_preserved_in_representation":
        normalisation_integrity.get(
            "divider_pages_3_7_preserved_in_representation"
        ),

    "complete_10_page_representation_retained":
        normalisation_integrity.get(
            "complete_10_page_representation_retained"
        ),

    "source_scope_filtering_applied":
        normalisation_integrity.get(
            "source_scope_filtering_applied"
        ),

    "page_removal_applied":
        normalisation_integrity.get(
            "page_removal_applied"
        ),

    "page_cropping_applied":
        normalisation_integrity.get(
            "page_cropping_applied"
        ),

    "branch_B_structural_conversion_inherited":
        normalisation_integrity.get(
            "branch_B_structural_conversion_inherited"
        ),

    "branch_B_regeneration_attempted":
        normalisation_integrity.get(
            "branch_B_regeneration_attempted"
        ),

    "ocr_applied":
        normalisation_integrity.get(
            "ocr_applied"
        ),

    "chart_value_reconstruction_applied":
        normalisation_integrity.get(
            "chart_value_reconstruction_applied"
        ),

    "table_value_reconstruction_applied":
        normalisation_integrity.get(
            "table_value_reconstruction_applied"
        ),

    "timeline_reconstruction_applied":
        normalisation_integrity.get(
            "timeline_reconstruction_applied"
        ),

    "unicode_nfkc_normalisation_applied":
        normalisation_integrity.get(
            "unicode_nfkc_normalisation_applied"
        ),

    "unicode_space_standardisation_applied":
        normalisation_integrity.get(
            "unicode_space_standardisation_applied"
        ),

    "apostrophe_standardisation_applied":
        normalisation_integrity.get(
            "apostrophe_standardisation_applied"
        ),

    "dash_and_minus_standardisation_applied":
        normalisation_integrity.get(
            "dash_and_minus_standardisation_applied"
        ),

    "soft_hyphen_removal_applied":
        normalisation_integrity.get(
            "soft_hyphen_removal_applied"
        ),

    "line_endings_standardised":
        normalisation_integrity.get(
            "line_endings_standardised"
        ),

    "horizontal_whitespace_normalisation_applied":
        normalisation_integrity.get(
            "horizontal_whitespace_normalisation_applied"
        ),

    "semantic_harmonisation_applied":
        normalisation_integrity.get(
            "semantic_harmonisation_applied"
        ),

    "semantic_rewriting_applied":
        normalisation_integrity.get(
            "semantic_rewriting_applied"
        ),

    "unit_conversion_applied":
        normalisation_integrity.get(
            "unit_conversion_applied"
        ),

    "monetary_rescaling_applied":
        normalisation_integrity.get(
            "monetary_rescaling_applied"
        ),

    "numeric_calculation_applied":
        normalisation_integrity.get(
            "numeric_calculation_applied"
        ),

    "manual_reconstruction_applied":
        normalisation_integrity.get(
            "manual_reconstruction_applied"
        ),

    "manual_correction_applied":
        normalisation_integrity.get(
            "manual_correction_applied"
        ),

    "reference_values_used_for_transformation":
        normalisation_integrity.get(
            "reference_values_used_for_transformation"
        ),
}


print("Branch C representation integrity:")
print(
    json.dumps(
        representation_integrity,
        ensure_ascii=False,
        indent=2,
    )
)

Branch C representation integrity:
{
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "page_sequence_preserved": true,
  "deterministic_representation_verified": true,
  "critical_marker_checks": {
    "presentation_title": true,
    "safe_harbor": true,
    "management_commentary": true,
    "quarterly_business_performance": true,
    "net_revenue": true,
    "profit_and_loss_statement": true,
    "revenue_from_operations": true,
    "company_profile": true,
    "corporate_timeline": true,
    "operational_footprint": true
  },
  "all_critical_markers_preserved": true,
  "qualified_source_value_checks": {
    "distributors": true,
    "fabric": true,
    "stores": true,
    "retail_space": true,
    "apparel": true,
    "customers": true
  },
  "qualified_source_values_preserved": true,
  "token_preservation": {
    "percentages": {
      "count_before": 42,
      "count_after": 42,
      "missing_token_count": 0,
      "added_to

In [ ]:
# ============================================================
# 11. Final Branch C validation summary
# ============================================================

schema_diagnostics = {
    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_correct":
        bool(document_id_correct),

    "branch_correct":
        bool(branch_correct),

    "records_is_list":
        bool(records_is_list),

    "record_schema_valid":
        bool(extraction_schema_exact),

    "field_types_valid":
        bool(extraction_types_valid),

    "branch_C_structure_valid":
        bool(branch_c_structure_valid),

    "schema_validity":
        bool(schema_validity),
}


content_diagnostics = {
    "reference_record_count_valid":
        bool(reference_record_count_valid),

    "reference_category_counts_valid":
        bool(reference_category_counts_valid),

    "extraction_record_count_valid":
        bool(extraction_record_count_valid),

    "extraction_category_counts_valid":
        bool(extraction_category_counts_valid),

    "branch_C_scope_complete":
        structure_check.get(
            "scope_complete"
        ),

    "branch_C_content_diagnostics":
        structure_check.get(
            "content_diagnostics"
        ),

    "reference_identity_unique":
        bool(
            reference_duplicate_identity_count
            == 0
        ),

    "extraction_duplicate_identity_count":
        int(
            extraction_duplicate_identity_count
        ),

    "strict_alignment_count":
        int(
            strict_alignment_count
        ),

    "fallback_alignment_count":
        int(
            fallback_alignment_count
        ),

    "alignment_score_threshold":
        float(
            ALIGNMENT_SCORE_THRESHOLD
        ),

    "alignment_weights":
        ALIGNMENT_WEIGHTS,

    "remaining_missing_record_count":
        int(
            len(
                missing_reference_indices
            )
        ),

    "remaining_unsupported_record_count":
        int(
            len(
                unsupported_extraction_indices
            )
        ),
}


VALIDATION_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        INPUT_REPRESENTATION,

    "reference_records":
        int(
            len(reference_df)
        ),

    "extracted_records":
        int(
            len(extracted_df)
        ),

    "aligned_records":
        int(
            aligned_records
        ),

    "fully_correct_records":
        int(
            fully_correct_records
        ),

    "discrepant_records":
        int(
            discrepant_records
        ),

    "missing_records":
        int(
            missing_records
        ),

    "unsupported_extracted_records":
        int(
            unsupported_records
        ),

    "completeness":
        float(
            completeness
        ),

    "missing_rate":
        float(
            missing_records
            / len(reference_df)
            if len(reference_df)
            else 0.0
        ),

    "record_precision_exact":
        float(
            record_precision_exact
        ),

    "record_recall_exact":
        float(
            record_recall_exact
        ),

    "record_f1_exact":
        float(
            record_f1_exact
        ),

    "unsupported_rate":
        float(
            unsupported_records
            / len(extracted_df)
            if len(extracted_df)
            else 0.0
        ),

    "discrepancy_rate_among_aligned":
        float(
            discrepant_records
            / aligned_records
            if aligned_records
            else 0.0
        ),

    "overall_primary_field_accuracy":
        float(
            overall_primary_field_accuracy
        ),

    "primary_field_accuracy":
        primary_field_accuracy,

    "diagnostic_field_accuracy":
        diagnostic_field_accuracy,

    "schema_validity":
        bool(
            schema_validity
        ),

    "schema_diagnostics":
        schema_diagnostics,

    "content_diagnostics":
        content_diagnostics,

    "branch_C_representation_integrity":
        representation_integrity,

    "matching_rules": {
        "blocking_field":
            "Category",

        "strict_identity_fields": [
            "Category",
            "Topic",
            "Reporting Period",
        ],

        "fallback_identity_fields": [
            "Topic",
            "Reporting Period",
        ],

        "one_to_one_assignment":
            (
                "Strict identity matching followed by "
                "Category-blocked Hungarian one-to-one assignment "
                "using Topic and Reporting Period only."
            ),

        "matching_score_threshold":
            ALIGNMENT_SCORE_THRESHOLD,

        "matching_score_weights":
            ALIGNMENT_WEIGHTS,

        "value_used_for_alignment":
            False,

        "unit_used_for_alignment":
            False,

        "description_used_for_alignment":
            False,

        "source_location_used_for_alignment":
            False,

        "matching_rules_frozen_from_branch_A":
            True,
    },

    "comparison_rules": {
        "raw_extraction_modified":
            False,

        "manual_correction_applied":
            False,

        "comparison_normalisation_scope":
            "Comparison copies only",

        "identity_fields":
            IDENTITY_FIELDS,

        "primary_correctness_fields":
            PRIMARY_CORRECTNESS_FIELDS,

        "description":
            (
                "Normalised exact diagnostic only; excluded from "
                "primary exact-record correctness because the prompt "
                "permits a concise source-grounded description."
            ),

        "numeric_value":
            (
                "Exact represented numeric equality after deterministic "
                "parsing; no tolerance beyond floating-point representation."
            ),

        "qualified_value":
            (
                "Exact normalised textual agreement with source qualification "
                "preserved; no conversion of ~, +, or 'and counting' to "
                "exact numbers."
            ),

        "text_value":
            (
                "Conservative normalised exact textual agreement."
            ),

        "unit":
            (
                "Controlled notation equivalence only; no monetary "
                "rescaling or unit conversion."
            ),

        "source_location":
            "Exact physical PDF page agreement.",

        "d11_equivalence_rules_status":
            (
                "Revised D11 Branch A identity alignment and "
                "document/schema-level comparison rules reused "
                "unchanged for Branch C. No Branch-B-specific "
                "performance-driven equivalence rules were added."
            ),

        "equivalence_rules_frozen":
            True,
    },

    "reference_integrity_confirmation": {
        "reference_semantics_valid":
            bool(
                reference_semantics_valid
            ),

        "checks":
            reference_semantic_checks,

        "reference_modified_by_validation":
            False,
    },

    "category_metrics":
        category_metrics,

    "input_provenance": {
        "reference_file":
            REFERENCE_PATH.name,

        "reference_sha256":
            REFERENCE_SHA256,

        "parsed_extraction_file":
            EXTRACTION_PATH.name,

        "parsed_extraction_sha256":
            EXTRACTION_SHA256,

        "structure_check_file":
            STRUCTURE_PATH.name,

        "structure_check_sha256":
            STRUCTURE_SHA256,

        "experiment_metadata_file":
            METADATA_PATH.name,

        "experiment_metadata_sha256":
            METADATA_SHA256,

        "normalisation_integrity_file":
            NORMALISATION_INTEGRITY_PATH.name,

        "normalisation_integrity_sha256":
            NORMALISATION_INTEGRITY_SHA256,

        "experiment_summary_file": (
            EXPERIMENT_SUMMARY_PATH.name
            if EXPERIMENT_SUMMARY_PATH.exists()
            else None
        ),

        "experiment_summary_sha256":
            EXPERIMENT_SUMMARY_SHA256,

        "branch_C_structure_valid":
            bool(
                branch_c_structure_valid
            ),

        "parsed_extraction_hash_matches_metadata":
            bool(
                parsed_extraction_hash_matches_metadata
            ),

        "source_hash_matches_stage_1":
            bool(
                source_hash_matches_stage_1
            ),

        "parent_B_equivalence_passed":
            bool(
                parent_equivalence_passed
            ),
    },

    "comparison_rules_frozen_from_branch_A":
        True,

    "validation_timestamp":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


print(
    json.dumps(
        VALIDATION_SUMMARY,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "document_id": "D11",
  "document_name": "Siyaram Silk Mills Limited — Investor Presentation Q4 & FY24",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised page-aware structural Markdown",
  "reference_records": 199,
  "extracted_records": 163,
  "aligned_records": 147,
  "fully_correct_records": 122,
  "discrepant_records": 25,
  "missing_records": 52,
  "unsupported_extracted_records": 16,
  "completeness": 0.7386934673366834,
  "missing_rate": 0.2613065326633166,
  "record_precision_exact": 0.7484662576687117,
  "record_recall_exact": 0.6130653266331658,
  "record_f1_exact": 0.6740331491712708,
  "unsupported_rate": 0.09815950920245399,
  "discrepancy_rate_among_aligned": 0.17006802721088435,
  "overall_primary_field_accuracy": 0.9410430839002268,
  "primary_field_accuracy": {
    "Value": 0.8435374149659864,
    "Unit": 0.9795918367346939,
    "Source Location": 1.0
  },
  "diagnostic_field_accuracy"

In [ ]:
# ============================================================
# 12. Export reproducible Validation C outputs
# ============================================================

SUMMARY_PATH = (
    OUTPUT_DIR
    / "D11_branch_C_validation_summary.json"
)

DETAILED_PATH = (
    OUTPUT_DIR
    / "D11_branch_C_validation_detailed.csv"
)

FULLY_CORRECT_PATH = (
    OUTPUT_DIR
    / "D11_branch_C_fully_correct_records.csv"
)

DISCREPANT_PATH = (
    OUTPUT_DIR
    / "D11_branch_C_discrepant_records.csv"
)

MISSING_PATH = (
    OUTPUT_DIR
    / "D11_branch_C_missing_records.csv"
)

UNSUPPORTED_PATH = (
    OUTPUT_DIR
    / "D11_branch_C_unsupported_records.csv"
)

FIELD_VALIDATION_PATH = (
    OUTPUT_DIR
    / "D11_branch_C_field_validation.csv"
)

CATEGORY_METRICS_PATH = (
    OUTPUT_DIR
    / "D11_branch_C_category_metrics.csv"
)

REFERENCE_SEMANTICS_PATH = (
    OUTPUT_DIR
    / "D11_reference_semantics_confirmation.json"
)

ALIGNMENT_ISSUES_PATH = (
    OUTPUT_DIR
    / "D11_branch_C_alignment_issues.json"
)

VALIDATION_METADATA_PATH = (
    OUTPUT_DIR
    / "D11_branch_C_validation_metadata.json"
)

VALIDATION_CONCLUSION_PATH = (
    OUTPUT_DIR
    / "D11_branch_C_validation_conclusion.json"
)


fully_correct_records_df = (
    comparison_df.loc[
        comparison_df[
            "Fully Correct Primary Record"
        ]
    ].copy()
)

field_validation_df = pd.DataFrame([
    {
        "Field": field,

        "Role": (
            "Primary correctness"
            if field in PRIMARY_CORRECTNESS_FIELDS
            else "Identity/diagnostic"
        ),

        "Correct":
            int(
                comparison_df[
                    f"{field} Correct"
                ].sum()
            ),

        "Compared":
            int(aligned_records),

        "Accuracy": (
            float(
                comparison_df[
                    f"{field} Correct"
                ].mean()
            )
            if aligned_records
            else 0.0
        ),
    }
    for field in FIELDS
])

category_metrics_df = pd.DataFrame([
    {
        "Category":
            category,
        **metrics,
    }
    for category, metrics
    in category_metrics.items()
])


VALIDATION_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "validation_type":
        "Post-extraction reference-value agreement",

    "raw_extraction_modified":
        False,

    "manual_correction_applied":
        False,

    "comparison_normalisation_scope":
        "Comparison copies only",

    "comparison_rules_frozen_from_branch_A":
        True,

    "created_at":
        datetime.now(timezone.utc).isoformat(),

    "python_version":
        sys.version,

    "platform":
        platform.platform(),
}

validation_status = (
    "Completed without discrepancies"
    if (
        fully_correct_records == len(reference_df)
        and missing_records == 0
        and unsupported_records == 0
        and schema_validity
    )
    else "Completed with discrepancies"
)

VALIDATION_CONCLUSION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "validation_status":
        validation_status,

    "reference_records":
        int(len(reference_df)),

    "extracted_records":
        int(len(extracted_df)),

    "aligned_records":
        int(aligned_records),

    "fully_correct_records":
        int(fully_correct_records),

    "discrepant_records":
        int(discrepant_records),

    "missing_records":
        int(missing_records),

    "unsupported_extracted_records":
        int(unsupported_records),

    "completeness":
        float(completeness),

    "record_precision_exact":
        float(record_precision_exact),

    "record_recall_exact":
        float(record_recall_exact),

    "record_f1_exact":
        float(record_f1_exact),

    "overall_primary_field_accuracy":
        float(overall_primary_field_accuracy),

    "schema_valid":
        bool(schema_validity),

    "normalisation_integrity_passed":
        bool(
            representation_integrity[
                "normalisation_integrity_passed"
            ]
        ),

    "comparison_rules_frozen_from_branch_A":
        True,
}


SUMMARY_PATH.write_text(
    json.dumps(
        VALIDATION_SUMMARY,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

comparison_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig",
)

fully_correct_records_df.to_csv(
    FULLY_CORRECT_PATH,
    index=False,
    encoding="utf-8-sig",
)

discrepant_records_df.to_csv(
    DISCREPANT_PATH,
    index=False,
    encoding="utf-8-sig",
)

missing_records_df.to_csv(
    MISSING_PATH,
    index=False,
    encoding="utf-8-sig",
)

unsupported_records_df.to_csv(
    UNSUPPORTED_PATH,
    index=False,
    encoding="utf-8-sig",
)

field_validation_df.to_csv(
    FIELD_VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig",
)

category_metrics_df.to_csv(
    CATEGORY_METRICS_PATH,
    index=False,
    encoding="utf-8-sig",
)

REFERENCE_SEMANTICS_PATH.write_text(
    json.dumps(
        {
            "document_id":
                DOCUMENT_ID,

            "reference_semantics_valid":
                reference_semantics_valid,

            "checks":
                reference_semantic_checks,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

ALIGNMENT_ISSUES_PATH.write_text(
    json.dumps(
        {
            "alignment_score_threshold":
                ALIGNMENT_SCORE_THRESHOLD,

            "alignment_weights":
                ALIGNMENT_WEIGHTS,

            "strict_alignment_count":
                strict_alignment_count,

            "fallback_alignment_count":
                fallback_alignment_count,

            "controlled_fallback_alignments":
                alignment_diagnostics,

            "missing_reference_indices":
                missing_reference_indices,

            "unsupported_extraction_indices":
                unsupported_extraction_indices,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

VALIDATION_METADATA_PATH.write_text(
    json.dumps(
        VALIDATION_METADATA,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

VALIDATION_CONCLUSION_PATH.write_text(
    json.dumps(
        VALIDATION_CONCLUSION,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# Final methodology/provenance assertions
# ------------------------------------------------------------

assert reference_semantics_valid
assert branch_c_structure_valid
assert parsed_extraction_hash_matches_metadata
assert source_hash_matches_stage_1
assert parent_equivalence_passed
assert representation_integrity[
    "normalisation_integrity_passed"
]

assert (
    aligned_records
    + missing_records
    == len(reference_df)
)

assert (
    aligned_records
    + unsupported_records
    == len(extracted_df)
)

assert (
    fully_correct_records
    + discrepant_records
    == aligned_records
)

required_outputs = [
    SUMMARY_PATH,
    DETAILED_PATH,
    FULLY_CORRECT_PATH,
    DISCREPANT_PATH,
    MISSING_PATH,
    UNSUPPORTED_PATH,
    FIELD_VALIDATION_PATH,
    CATEGORY_METRICS_PATH,
    REFERENCE_SEMANTICS_PATH,
    ALIGNMENT_ISSUES_PATH,
    VALIDATION_METADATA_PATH,
    VALIDATION_CONCLUSION_PATH,
]

missing_outputs = [
    path.name
    for path in required_outputs
    if not path.exists()
]

if missing_outputs:
    raise AssertionError(
        f"Missing output files: {missing_outputs}"
    )

print("Validation B — D11 completed successfully.")
print("Validation status:", validation_status)
print("Reference records:", len(reference_df))
print("Extracted records:", len(extracted_df))
print("Aligned records:", aligned_records)
print("Fully correct records:", fully_correct_records)
print("Discrepant records:", discrepant_records)
print("Missing records:", missing_records)
print("Unsupported/unmatched records:", unsupported_records)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1_exact, 4))
print(
    "Primary field accuracy:",
    round(overall_primary_field_accuracy, 4),
)
print("Schema valid:", schema_validity)

if not schema_validity:
    print(
        "WARNING: Schema validity is False. "
        "The result is retained as an experimental outcome."
    )

print(
    "Conversion integrity passed:",
    representation_integrity[
        "normalisation_integrity_passed"
    ],
)
print("Comparison rules frozen from Branch A:", True)

for path in required_outputs:
    print("-", path.name)

for path in required_outputs:
    files.download(path)

Validation B — D11 completed successfully.
Validation status: Completed with discrepancies
Reference records: 199
Extracted records: 163
Aligned records: 147
Fully correct records: 122
Discrepant records: 25
Missing records: 52
Unsupported/unmatched records: 16
Completeness: 0.7387
Exact F1: 0.674
Primary field accuracy: 0.941
Schema valid: True
Conversion integrity passed: True
Comparison rules frozen from Branch A: True
- D11_branch_C_validation_summary.json
- D11_branch_C_validation_detailed.csv
- D11_branch_C_fully_correct_records.csv
- D11_branch_C_discrepant_records.csv
- D11_branch_C_missing_records.csv
- D11_branch_C_unsupported_records.csv
- D11_branch_C_field_validation.csv
- D11_branch_C_category_metrics.csv
- D11_reference_semantics_confirmation.json
- D11_branch_C_alignment_issues.json
- D11_branch_C_validation_metadata.json
- D11_branch_C_validation_conclusion.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>